# BOB Theta evaluation — Optuna GPSampler revision

This notebook preserves the original BOB signal-processing, calibration/test split, objective, search bounds, evaluation budget, and reported metrics. The primary change is the Bayesian-optimization driver.

- Default: `OPTIMIZERS_TO_RUN = ('optuna_gp',)` runs Optuna's `GPSampler` for 100 trials, including 20 startup trials.
- Side-by-side: change it to `('optuna_gp', 'skopt_gp')`. Both optimizers receive the same objective, bounds, budget, and seed; this approximately doubles optimization work.
- Backward compatibility: the top-level `best_params`, `opt_metrics_*`, and `opt_errors_*` fields still refer to `PRIMARY_OPTIMIZER`. Detailed results and convergence histories for every enabled optimizer are stored under `optimizer_runs`.

The convergence history records each calibration-objective value and the best value so far. The final plot is calibration convergence only; held-out test metrics remain the accuracy comparison.


In [ ]:
# BOB Theta — Evaluation (native 300 Hz)
# Conditions saved: DEFAULT (pre-optimization Burg) + OPT (Bayesian-optimized, Opt_o)
# Primary optimizer: Optuna GPSampler. Optional historical baseline: skopt gp_minimize.
# Metrics: MACE, PLV, CSD (circular SD), Acc. GT = matched zero-phase FIR on the same channel.
import os

# Optuna GPSampler uses torch internally. This must be set before torch is imported.
os.environ.setdefault('OMP_NUM_THREADS', '1')

import gc, glob, json, random, subprocess, sys, time
import importlib.util
import numpy as np
from scipy.signal import butter, hilbert, lfilter, lfilter_zi, filtfilt, welch, firwin
from scipy.stats import pearsonr, spearmanr
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# Colab already includes scipy and torch in its standard runtime. Install only what is missing
# or needs a current GPSampler. scikit-optimize remains available for the historical baseline.
requirements = ['optuna>=4.0', 'scikit-optimize>=0.10.2']
if importlib.util.find_spec('torch') is None:
    requirements.append('torch')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *requirements])

import optuna
import skopt
from optuna.samplers import GPSampler
from skopt import gp_minimize
from skopt.space import Integer, Real

optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"setup ok | optuna={optuna.__version__} | skopt={skopt.__version__}")


In [ ]:
# ---- CONFIG ----
STIM_DIR = '/content/drive/MyDrive/RAIN_IRC/data splits/onset'
OUTDIR   = '/content/drive/MyDrive/RAIN_IRC/results/BOB_onset'
os.makedirs(OUTDIR, exist_ok=True)

EVAL_TIMES_S = [0.3, 0.4, 0.5, 0.6, 0.7]
HORIZON_MS   = 0.0
GT_LOW, GT_HIGH = 4.0, 8.0            # fixed theta GT band (both conditions scored against this)
GT_FIR_NUMTAPS  = 175                 # 300 Hz (half of the 351 used at 600 Hz)
AR_CONTEXT_DIV  = 2
THETA_SEARCH_LO, THETA_SEARCH_HI = 4.0, 8.0

# ---- OPTIMIZER SETTINGS (the main comparison controls) ----
PRIMARY_OPTIMIZER = 'optuna_gp'
OPTIMIZERS_TO_RUN = ('optuna_gp',)     # use ('optuna_gp', 'skopt_gp') for side-by-side runs
N_BAYES_CALLS = 100                    # total objective evaluations per optimizer and subject
N_INITIAL = 20                         # startup/random evaluations included in N_BAYES_CALLS
SEED = 42
GPSAMPLER_DETERMINISTIC_OBJECTIVE = True
LOG_EVERY_N_TRIALS = 1                 # 0 disables per-trial console logging
CONVERGENCE_TOLERANCE_DEG = 1.0
RESULTS_FILENAME = 'bob_theta_results_optuna_gp.json'
CONVERGENCE_FIGURE_FILENAME = 'bob_theta_optimizer_convergence.png'

SUPPORTED_OPTIMIZERS = {'optuna_gp', 'skopt_gp'}
assert OPTIMIZERS_TO_RUN, 'enable at least one optimizer'
assert PRIMARY_OPTIMIZER in OPTIMIZERS_TO_RUN
assert set(OPTIMIZERS_TO_RUN) <= SUPPORTED_OPTIMIZERS
assert 0 <= N_INITIAL < N_BAYES_CALLS
random.seed(SEED)
np.random.seed(SEED)

# Baseline (pre-optimization) Burg config — literature defaults, kept for default-vs-opt.
DEFAULT = {'ar_order': 16, 'context_s': 1.0, 'filt_low': 4.0, 'filt_high': 8.0,
           'filt_order': 2, 'hilbert_pad': 60}

# Single source of truth for both Optuna and skopt bounds.
# AR bounds were tuned at 600 Hz (~100 samp/cycle); at 300 Hz a cycle is ~50 samp, so these
# orders span more cycles now -- watch for AR saturating the ceiling and scale down if so.
PARAM_BOUNDS = {
    'ar_order': (24, 100),
    'context_s': (0.8, 1.6),
    'filt_low': (4.0, 5.5),
    'filt_high': (6.0, 8.0),
    'filt_order': (1, 4),
    'hilbert_pad': (30, 100),
}
SEARCH_SPACE = [
    Integer(*PARAM_BOUNDS['ar_order'], name='ar_order'),
    Real(*PARAM_BOUNDS['context_s'], name='context_s'),
    Real(*PARAM_BOUNDS['filt_low'], name='filt_low'),
    Real(*PARAM_BOUNDS['filt_high'], name='filt_high'),
    Integer(*PARAM_BOUNDS['filt_order'], name='filt_order'),
    Integer(*PARAM_BOUNDS['hilbert_pad'], name='hilbert_pad'),
]

files = sorted(glob.glob(os.path.join(STIM_DIR, '*.npz')))
assert files, f"no npz found in {STIM_DIR}"
_d0 = np.load(files[0], allow_pickle=True)
FS = float(_d0['fs']); PRE_SAMP = int(_d0['onset_in_epoch']); _d0.close()
HORIZON_SAMP = int(HORIZON_MS * FS / 1000.0)
EVAL_TIMES_SAMP = [PRE_SAMP + int(t * FS) for t in EVAL_TIMES_S]
print(f"FS={FS:.0f} Hz | t=0 at sample {PRE_SAMP} | GT {GT_LOW}-{GT_HIGH} Hz FIR-{GT_FIR_NUMTAPS}")
print(f"eval samples {EVAL_TIMES_SAMP} | {len(files)} subject files")
print(f"optimizers={OPTIMIZERS_TO_RUN} | budget={N_BAYES_CALLS} | startup={N_INITIAL} | seed={SEED}")


In [ ]:
# ---- ENGINE ----
def make_causal_filter(lo, hi, order):
    nyq = FS / 2.0
    lo = max(0.5, min(lo, nyq - 1)); hi = max(lo + 0.5, min(hi, nyq - 0.5))
    return butter(int(order), [lo / nyq, hi / nyq], btype='band')

def causal_filter_signal(sig1d, b, a):
    zi = lfilter_zi(b, a) * sig1d[0]
    out, _ = lfilter(b, a, sig1d, zi=zi)
    return out.astype(np.float32)

def burg_ar_coeffs(x, order):
    x = x.astype(np.float64); n = len(x); order = int(order)
    if n <= order + 1: return np.zeros(order)
    ef = x.copy(); eb = x.copy(); a = np.zeros(order)
    for m in range(order):
        ef_s, eb_s = ef[m + 1:n], eb[m:n - 1]; L = min(len(ef_s), len(eb_s))
        if L <= 0: break
        ef_s, eb_s = ef_s[:L].copy(), eb_s[:L].copy()
        num = -2.0 * np.dot(ef_s, eb_s); den = np.dot(ef_s, ef_s) + np.dot(eb_s, eb_s)
        if den < 1e-12: break
        k = np.clip(num / den, -0.9999, 0.9999)
        a_new = np.zeros(m + 1); a_new[m] = k
        if m > 0: a_new[:m] = a[:m] + k * a[m - 1::-1]
        a[:m + 1] = a_new
        ef[m + 1:m + 1 + L] = ef_s + k * eb_s; eb[m + 1:m + 1 + L] = eb_s + k * ef_s; n = m + 1 + L
    return -a

def ar_extrapolate(x, coeffs, n_steps):
    p = len(coeffs); coeffs = np.asarray(coeffs, np.float64)
    out = np.empty(len(x) + n_steps, np.float64); out[:len(x)] = x
    for i in range(len(x), len(x) + n_steps):
        out[i] = out[i - 1] if i < p else np.dot(coeffs, out[i - p:i][::-1])
    return out

def burg_predict_phase(csig, predict_at, horizon, ar_order, context_s, hpad):
    csamp = int(context_s * FS); pad = int(hpad)
    start = max(0, predict_at - csamp + 1)
    x = csig[start:predict_at + 1].astype(np.float64)
    ar_order = min(int(ar_order), len(x) // AR_CONTEXT_DIV)
    if len(x) < ar_order + 5 or np.std(x) < 1e-10: return 0.0, 1.0
    ext = ar_extrapolate(x, burg_ar_coeffs(x, ar_order), horizon + pad)
    tgt = len(x) - 1 + horizon
    if tgt >= len(ext): return 0.0, 1.0
    ph = np.angle(hilbert(ext)[tgt]); return np.sin(ph), np.cos(ph)

def compute_gt(raw_epoch, lo, hi):
    nyq = FS / 2.0
    lo = max(0.5, min(lo, nyq - 1)); hi = max(lo + 0.5, min(hi, nyq - 0.5))
    taps = firwin(GT_FIR_NUMTAPS, [lo, hi], pass_zero=False, fs=FS)
    ph = np.angle(hilbert(filtfilt(taps, [1.0], raw_epoch.astype(np.float64))))
    return np.sin(ph).astype(np.float32), np.cos(ph).astype(np.float32)

def mace_from_errors(e):
    return 90.0 if len(e) < 10 else float(np.degrees(np.mean(np.abs(e))))

def errors_to_metrics(e):
    if len(e) < 10:
        return {'mace': 90.0, 'plv': 0.0, 'csd': 180.0, 'acc': 0.0, 'bias': 0.0, 'n_samples': 0}
    mean_abs = float(np.mean(np.abs(e))); R = float(np.abs(np.mean(np.exp(1j * e))))
    return {'mace': float(np.degrees(mean_abs)), 'plv': R,
            'csd': float(np.degrees(np.sqrt(-2.0 * np.log(max(R, 1e-12))))),   # circular SD (deg)
            'acc': float((1.0 - mean_abs / np.pi) * 100.0),
            'bias': float(np.degrees(np.angle(np.mean(np.exp(1j * e))))),
            'n_samples': int(len(e))}

def collect_errors(raw, gs_l, gc_l, trials, ar, ctx, lo, hi, ordf, hpad):
    try: b, a = make_causal_filter(lo, hi, int(ordf))
    except Exception: return np.array([])
    L = raw.shape[1]; errs = []
    for tr in trials:
        cs = causal_filter_signal(raw[tr], b, a)
        for es in EVAL_TIMES_SAMP:
            g = es + HORIZON_SAMP
            if g >= L: continue
            s, c = burg_predict_phase(cs, es, HORIZON_SAMP, int(ar), ctx, int(hpad))
            gsv, gcv = gs_l[tr][g], gc_l[tr][g]
            errs.append(np.arctan2(s * gcv - c * gsv, c * gcv + s * gsv))
    return np.array(errs)

def collect_errors_per_tp(raw, gs_l, gc_l, trials, ar, ctx, lo, hi, ordf, hpad):
    try: b, a = make_causal_filter(lo, hi, int(ordf))
    except Exception: return {t: np.array([]) for t in EVAL_TIMES_S}
    L = raw.shape[1]; by_t = {t: [] for t in EVAL_TIMES_S}
    for tr in trials:
        cs = causal_filter_signal(raw[tr], b, a)
        for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP):
            g = es + HORIZON_SAMP
            if g >= L: continue
            s, c = burg_predict_phase(cs, es, HORIZON_SAMP, int(ar), ctx, int(hpad))
            gsv, gcv = gs_l[tr][g], gc_l[tr][g]
            by_t[t].append(np.arctan2(s * gcv - c * gsv, c * gcv + s * gsv))
    return {t: np.array(v) for t, v in by_t.items()}

def run_per_tp(raw, gs_l, gc_l, trials, p):
    return collect_errors_per_tp(raw, gs_l, gc_l, trials, p['ar_order'], p['context_s'],
                                 p['filt_low'], p['filt_high'], p['filt_order'], p['hilbert_pad'])

def itpc_from_gt(gs_l, gc_l, trials, esamp):
    ph = np.array([np.arctan2(gs_l[tr][esamp], gc_l[tr][esamp]) for tr in trials])
    return float(np.abs(np.mean(np.exp(1j * ph))))

def estimate_theta_peak(raw, onset_samp):
    seg = raw[:, onset_samp:]
    if seg.shape[1] < 16: return np.nan
    nfft = int(2 ** np.ceil(np.log2(seg.shape[1] * 4))); psd = None
    for tr in range(seg.shape[0]):
        f, px = welch(seg[tr].astype(np.float64), fs=FS, nperseg=seg.shape[1], nfft=nfft, detrend='linear')
        psd = px if psd is None else psd + px
    psd /= seg.shape[0]; band = (f >= THETA_SEARCH_LO) & (f <= THETA_SEARCH_HI)
    return float(f[band][np.argmax(psd[band])]) if np.any(band) else np.nan


# ---- OPTIMIZER ADAPTERS ----
# Both adapters call the same list-valued objective used by the original notebook.
PARAMETER_ORDER = ('ar_order', 'context_s', 'filt_low', 'filt_high', 'filt_order', 'hilbert_pad')

def normalized_params(params, clip_effective_ar=True):
    p = {
        'ar_order': int(params['ar_order']),
        'context_s': float(params['context_s']),
        'filt_low': float(params['filt_low']),
        'filt_high': float(params['filt_high']),
        'filt_order': int(params['filt_order']),
        'hilbert_pad': int(params['hilbert_pad']),
    }
    if clip_effective_ar:
        p['ar_order'] = min(p['ar_order'], int(p['context_s'] * FS) // AR_CONTEXT_DIV)
    return p

def evaluations_to_tolerance(history, tolerance_deg=1.0):
    if not history:
        return None
    threshold = history[-1]['best_objective'] + float(tolerance_deg)
    return next(row['trial'] for row in history if row['best_objective'] <= threshold)

def _maybe_log_trial(subject_name, optimizer_name, row):
    if LOG_EVERY_N_TRIALS and (row['trial'] == 1 or row['trial'] % LOG_EVERY_N_TRIALS == 0):
        print(f"  {subject_name} | {optimizer_name} | trial {row['trial']:>3}/{N_BAYES_CALLS} | "
              f"MACE={row['objective']:.3f} deg | best={row['best_objective']:.3f} deg | "
              f"objective={row['objective_seconds']:.2f}s")

def run_optimizer(optimizer_name, objective, subject_name):
    history = []
    optimizer_t0 = time.time()

    if optimizer_name == 'optuna_gp':
        sampler = GPSampler(
            seed=SEED,
            n_startup_trials=N_INITIAL,
            deterministic_objective=GPSAMPLER_DETERMINISTIC_OBJECTIVE,
        )
        study = optuna.create_study(
            direction='minimize', sampler=sampler, study_name=f'bob_theta_{subject_name}'
        )

        def optuna_objective(trial):
            b = PARAM_BOUNDS
            params = [
                trial.suggest_int('ar_order', *b['ar_order']),
                trial.suggest_float('context_s', *b['context_s']),
                trial.suggest_float('filt_low', *b['filt_low']),
                trial.suggest_float('filt_high', *b['filt_high']),
                trial.suggest_int('filt_order', *b['filt_order']),
                trial.suggest_int('hilbert_pad', *b['hilbert_pad']),
            ]
            t0 = time.time()
            value = float(objective(params))
            trial.set_user_attr('objective_seconds', float(time.time() - t0))
            return value

        def optuna_callback(study, trial):
            row = {
                'trial': int(trial.number + 1),
                'objective': float(trial.value),
                'best_objective': float(study.best_value),
                'objective_seconds': float(trial.user_attrs.get('objective_seconds', float('nan'))),
                'params': normalized_params(trial.params, clip_effective_ar=False),
            }
            history.append(row)
            _maybe_log_trial(subject_name, optimizer_name, row)

        study.optimize(
            optuna_objective,
            n_trials=N_BAYES_CALLS,
            callbacks=[optuna_callback],
            n_jobs=1,
            show_progress_bar=False,
        )
        suggested_best = normalized_params(study.best_params, clip_effective_ar=False)
        calibration_best = float(study.best_value)

    elif optimizer_name == 'skopt_gp':
        objective_seconds = []

        def timed_skopt_objective(params):
            t0 = time.time()
            value = float(objective(params))
            objective_seconds.append(float(time.time() - t0))
            return value

        def skopt_callback(result):
            idx = len(result.func_vals) - 1
            raw_params = dict(zip(PARAMETER_ORDER, result.x_iters[idx]))
            row = {
                'trial': int(idx + 1),
                'objective': float(result.func_vals[idx]),
                'best_objective': float(np.min(result.func_vals)),
                'objective_seconds': float(objective_seconds[idx]),
                'params': normalized_params(raw_params, clip_effective_ar=False),
            }
            history.append(row)
            _maybe_log_trial(subject_name, optimizer_name, row)
            return False

        result = gp_minimize(
            timed_skopt_objective,
            SEARCH_SPACE,
            n_calls=N_BAYES_CALLS,
            n_initial_points=N_INITIAL,
            random_state=SEED,
            verbose=False,
            callback=[skopt_callback],
        )
        suggested_best = normalized_params(dict(zip(PARAMETER_ORDER, result.x)), clip_effective_ar=False)
        calibration_best = float(result.fun)

    else:
        raise ValueError(f'unknown optimizer: {optimizer_name}')

    best_params = normalized_params(suggested_best, clip_effective_ar=True)
    return {
        'optimizer': optimizer_name,
        'seed': int(SEED),
        'n_evaluations': int(len(history)),
        'n_initial': int(N_INITIAL),
        'calibration_best_mace': calibration_best,
        'suggested_best_params': suggested_best,
        'best_params': best_params,
        'optimizer_total_seconds': float(time.time() - optimizer_t0),
        'evaluations_to_within_tolerance': evaluations_to_tolerance(
            history, CONVERGENCE_TOLERANCE_DEG
        ),
        'convergence_tolerance_deg': float(CONVERGENCE_TOLERANCE_DEG),
        'convergence_history': history,
    }


In [ ]:
# ---- RUN ----
print(f"{'Subject':<12}{'N':>5}{'defMACE':>8}{'optMACE':>8}{'optPLV':>7}{'optCSD':>7}{'optAcc':>7}{'SNR':>6}{'ITPC':>6}{'Peak':>6}")
print('-' * 80)
results = []
for fp in files:
    t0 = time.time(); d = np.load(fp, allow_pickle=True); name = os.path.basename(fp)[:-4]
    if float(d['fs']) != FS or int(d['onset_in_epoch']) != PRE_SAMP:
        print(f"{name}: geometry mismatch, skipping"); d.close(); continue
    epochs = d['epochs']; best = int(d['best_ch_idx']); ch_name = str(d['best_ch_name'])
    split_arr = np.array([str(x) for x in d['split']]); theta_snr = float(d['ch_theta_snr'][best])
    raw = epochs[:, best, :].astype(np.float32); n_tr = raw.shape[0]
    cal = np.where(split_arr == 'cal')[0]; test = np.where(split_arr == 'test')[0]

    gs_l, gc_l = [], []
    for tr in range(n_tr):
        s, c = compute_gt(raw[tr], GT_LOW, GT_HIGH); gs_l.append(s); gc_l.append(c)

    # --- baseline (pre-optimization) on TEST ---
    dflt_tp = run_per_tp(raw, gs_l, gc_l, test, DEFAULT)
    dflt_err = np.concatenate([dflt_tp[t] for t in EVAL_TIMES_S]) if any(len(dflt_tp[t]) for t in EVAL_TIMES_S) else np.array([])
    dflt_m = errors_to_metrics(dflt_err); dflt_tp_m = {t: errors_to_metrics(dflt_tp[t]) for t in EVAL_TIMES_S}

    # --- shared Bayesian-optimization objective on CAL (unchanged) ---
    def objective(params):
        ar, ctx, lo, hi, ordf, hpad = params
        if lo >= hi - 0.5: return 90.0
        arc = min(int(ar), int(ctx * FS) // AR_CONTEXT_DIV)
        if arc < 4: return 90.0
        try: make_causal_filter(lo, hi, int(ordf))
        except Exception: return 90.0
        return mace_from_errors(collect_errors(raw, gs_l, gc_l, cal, arc, ctx, lo, hi, int(ordf), int(hpad)))

    # Each enabled optimizer receives the same objective, bounds, budget, startup count, and seed.
    optimizer_runs = {}
    for optimizer_name in OPTIMIZERS_TO_RUN:
        run = run_optimizer(optimizer_name, objective, name)
        bp_i = run['best_params']

        # --- this optimizer's selected parameters on held-out TEST ---
        tp_i = run_per_tp(raw, gs_l, gc_l, test, bp_i)
        err_i = np.concatenate([tp_i[t] for t in EVAL_TIMES_S]) if any(len(tp_i[t]) for t in EVAL_TIMES_S) else np.array([])
        metrics_i = errors_to_metrics(err_i)
        tp_metrics_i = {t: errors_to_metrics(tp_i[t]) for t in EVAL_TIMES_S}
        run.update({
            'test_metrics_pooled': metrics_i,
            'test_metrics_per_timepoint': {str(t): tp_metrics_i[t] for t in EVAL_TIMES_S},
            'test_errors_per_timepoint_rad': {
                str(t): [float(x) for x in tp_i[t]] for t in EVAL_TIMES_S
            },
        })
        optimizer_runs[optimizer_name] = run

    # Preserve the original top-level schema using the selected primary optimizer.
    primary = optimizer_runs[PRIMARY_OPTIMIZER]
    bp = primary['best_params']
    opt_m = primary['test_metrics_pooled']
    opt_tp_m = primary['test_metrics_per_timepoint']
    opt_tp_errors = primary['test_errors_per_timepoint_rad']

    itpc_tp = {t: (itpc_from_gt(gs_l, gc_l, test, es) if es < raw.shape[1] else float('nan'))
               for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP)}
    itpc_mean = float(np.nanmean(list(itpc_tp.values()))); f_peak = estimate_theta_peak(raw, PRE_SAMP)

    results.append({
        'name': name, 'best_ch': ch_name, 'n_trials': int(n_tr),
        'n_cal': int(len(cal)), 'n_test': int(len(test)), 'theta_snr': theta_snr,
        'theta_itpc_per_timepoint': {str(t): itpc_tp[t] for t in EVAL_TIMES_S},
        'theta_itpc_mean': itpc_mean,
        'theta_peak_hz': float(f_peak) if np.isfinite(f_peak) else None,
        'primary_optimizer': PRIMARY_OPTIMIZER,
        'optimizer_runs': optimizer_runs,
        'best_params': bp, 'default_params': DEFAULT,
        'default_metrics_pooled': dflt_m,
        'default_metrics_per_timepoint': {str(t): dflt_tp_m[t] for t in EVAL_TIMES_S},
        'default_errors_per_timepoint_rad': {str(t): [float(x) for x in dflt_tp[t]] for t in EVAL_TIMES_S},
        'opt_metrics_pooled': opt_m,
        'opt_metrics_per_timepoint': opt_tp_m,
        'opt_errors_per_timepoint_rad': opt_tp_errors,
    })
    pk = f"{f_peak:.2f}" if np.isfinite(f_peak) else " n/a"
    print(f"{name:<12}{opt_m['n_samples']:>5}{dflt_m['mace']:>7.1f}°{opt_m['mace']:>7.1f}°{opt_m['plv']:>7.3f}"
          f"{opt_m['csd']:>6.1f}°{opt_m['acc']:>6.1f}%{theta_snr:>6.2f}{itpc_mean:>6.3f}{pk:>6}  "
          f"[{PRIMARY_OPTIMIZER}; {time.time()-t0:.0f}s]")
    for optimizer_name, run in optimizer_runs.items():
        if optimizer_name != PRIMARY_OPTIMIZER:
            tm = run['test_metrics_pooled']
            print(f"  comparison {optimizer_name}: CAL best={run['calibration_best_mace']:.2f}° | "
                  f"TEST MACE={tm['mace']:.2f}° | within {CONVERGENCE_TOLERANCE_DEG:g}° by "
                  f"trial {run['evaluations_to_within_tolerance']}")
    d.close(); del epochs, raw, gs_l, gc_l; gc.collect()
print(f"\n{len(results)} subjects done")


In [ ]:
# ---- SUMMARY + SAVE ----
n = len(results)
dM = np.array([r['default_metrics_pooled']['mace'] for r in results])
oM = np.array([r['opt_metrics_pooled']['mace'] for r in results])
oP = np.array([r['opt_metrics_pooled']['plv'] for r in results])
oC = np.array([r['opt_metrics_pooled']['csd'] for r in results])
oA = np.array([r['opt_metrics_pooled']['acc'] for r in results])
dA = np.array([r['default_metrics_pooled']['acc'] for r in results])
SNR = np.array([r['theta_snr'] for r in results]); IT = np.array([r['theta_itpc_mean'] for r in results])

print('=' * 72); print(f'BOB THETA — DEFAULT vs OPTIMIZED ({PRIMARY_OPTIMIZER})'); print('=' * 72)
print(f"  Default   MACE {dM.mean():.1f}°±{dM.std():.1f}   Acc {dA.mean():.1f}%")
print(f"  Optimized MACE {oM.mean():.1f}°±{oM.std():.1f}   PLV {oP.mean():.3f}   CSD {oC.mean():.1f}°   Acc {oA.mean():.1f}%")
gain = dM - oM
print(f"  Optimization gain: {gain.mean():+.1f}° MACE  ({int(np.sum(gain > 0))}/{n} improved)")
print(f"  (n={n} unique participants -- independent samples; group/inferential stats valid)")

# paired default-vs-optimized significance (per-subject, N independent participants, two-sided)
from scipy.stats import wilcoxon, ttest_rel, rankdata
diff = dM - oM; da = oA - dA   # >0 means optimization improved (MACE down / Acc up)
group_stats = {'n_participants': int(n), 'paired_test': 'two-sided',
               'primary_optimizer': PRIMARY_OPTIMIZER,
               'mean_mace_default': float(dM.mean()), 'mean_mace_opt': float(oM.mean()),
               'mean_mace_reduction': float(diff.mean()), 'median_mace_reduction': float(np.median(diff)),
               'n_improved': int(np.sum(diff > 0))}
print('\n--- Paired default vs optimized (per-subject MACE, test set) ---')
print(f"  mean dMACE {diff.mean():+.2f}°   median {np.median(diff):+.2f}°   ({int(np.sum(diff > 0))}/{n} improved)")
if n >= 3 and np.any(diff != 0):
    w_stat, w_p = wilcoxon(dM, oM)
    nz = diff[diff != 0]; rk = rankdata(np.abs(nz))
    rbc = float((rk[nz > 0].sum() - rk[nz < 0].sum()) / rk.sum())
    t_stat, t_p = ttest_rel(dM, oM)
    sd = diff.std(ddof=1); dz = float(diff.mean() / sd) if sd > 0 else float('nan')
    group_stats.update({'wilcoxon_W': float(w_stat), 'wilcoxon_p': float(w_p), 'rank_biserial_r': rbc,
                        'ttest_t': float(t_stat), 'ttest_p': float(t_p), 'cohen_dz': dz})
    print(f"  Wilcoxon signed-rank: W={w_stat:.1f}, p={w_p:.4g}  (rank-biserial r={rbc:+.3f})")
    print(f"  Paired t-test:        t={t_stat:+.2f}, p={t_p:.4g}  (Cohen's dz={dz:+.3f})")
    if np.any(da != 0):
        _, ap = wilcoxon(oA, dA); group_stats['accuracy_wilcoxon_p'] = float(ap)
        print(f"  Accuracy (opt-def):   median {np.median(da):+.2f} pts, Wilcoxon p={ap:.4g}")
else:
    group_stats['note'] = 'insufficient nonzero differences'
    print("  (insufficient nonzero differences for a paired test)")

print('\n--- Per-timepoint (default vs optimized) ---')
print(f"{'ms':>5}{'defMACE':>9}{'optMACE':>9}{'optPLV':>8}{'optCSD':>8}{'optAcc':>8}{'ITPC':>8}{'N':>7}")
for t in EVAL_TIMES_S:
    dm = np.array([r['default_metrics_per_timepoint'][str(t)]['mace'] for r in results])
    g = lambda k: np.array([r['opt_metrics_per_timepoint'][str(t)][k] for r in results])
    ti = np.array([r['theta_itpc_per_timepoint'][str(t)] for r in results])
    print(f"{int(t * 1000):>5}{dm.mean():>8.1f}°{g('mace').mean():>8.1f}°{g('plv').mean():>8.3f}"
          f"{g('csd').mean():>7.1f}°{g('acc').mean():>7.1f}%{ti.mean():>8.3f}{g('n_samples').mean():>7.0f}")

print('\n--- Accuracy (optimized) correlations across subjects ---')
if np.std(SNR) > 0:
    r1, p1 = pearsonr(SNR, oA); rs1, ps1 = spearmanr(SNR, oA)
    print(f"  Acc vs SNR : r={r1:+.3f} (p={p1:.4f})  rho={rs1:+.3f} (p={ps1:.4f})")
if np.std(IT) > 0:
    r2, p2 = pearsonr(IT, oA); rs2, ps2 = spearmanr(IT, oA)
    print(f"  Acc vs ITPC: r={r2:+.3f} (p={p2:.4f})  rho={rs2:+.3f} (p={ps2:.4f})")

print('\n--- Per-subject optimal params ---')
print(f"{'Subject':<12}{'AR':>4}{'Ctx':>5}{'Low':>5}{'High':>6}{'Ord':>4}{'Pad':>5}{'Peak':>6}")
for r in results:
    p = r['best_params']; pk = r['theta_peak_hz']
    print(f"{r['name']:<12}{p['ar_order']:>4}{p['context_s']:>5.2f}{p['filt_low']:>5.2f}"
          f"{p['filt_high']:>6.2f}{p['filt_order']:>4}{p['hilbert_pad']:>5}{(f'{pk:.2f}' if pk else 'n/a'):>6}")

optimizer_summary = {}
print('\n--- Optimizer comparison (CAL convergence + held-out TEST) ---')
for optimizer_name in OPTIMIZERS_TO_RUN:
    cal_best = np.array([r['optimizer_runs'][optimizer_name]['calibration_best_mace'] for r in results])
    test_mace = np.array([r['optimizer_runs'][optimizer_name]['test_metrics_pooled']['mace'] for r in results])
    n_tol = np.array([r['optimizer_runs'][optimizer_name]['evaluations_to_within_tolerance'] for r in results], dtype=float)
    elapsed = np.array([r['optimizer_runs'][optimizer_name]['optimizer_total_seconds'] for r in results])
    optimizer_summary[optimizer_name] = {
        'mean_calibration_best_mace': float(cal_best.mean()),
        'mean_test_mace': float(test_mace.mean()),
        'median_evaluations_to_within_tolerance': float(np.median(n_tol)),
        'convergence_tolerance_deg': float(CONVERGENCE_TOLERANCE_DEG),
        'mean_optimizer_total_seconds': float(elapsed.mean()),
    }
    print(f"  {optimizer_name:<10} CAL best={cal_best.mean():.2f}° | TEST MACE={test_mace.mean():.2f}° | "
          f"median evals within {CONVERGENCE_TOLERANCE_DEG:g}°={np.median(n_tol):.0f} | "
          f"mean total={elapsed.mean():.1f}s/subject")

out = {'model': 'BOB',
       'config': {'fs': FS, 'pre_samp': PRE_SAMP, 'eval_times_s': EVAL_TIMES_S, 'horizon_ms': HORIZON_MS,
                  'gt_band': [GT_LOW, GT_HIGH], 'gt_fir_numtaps': GT_FIR_NUMTAPS,
                  'primary_optimizer': PRIMARY_OPTIMIZER, 'optimizers_run': list(OPTIMIZERS_TO_RUN),
                  'n_bayes_calls': N_BAYES_CALLS, 'n_initial': N_INITIAL, 'seed': SEED,
                  'gpsampler_deterministic_objective': GPSAMPLER_DETERMINISTIC_OBJECTIVE,
                  'search_bounds': {k: list(v) for k, v in PARAM_BOUNDS.items()},
                  'split': 'chronological cal/test from npz split array', 'conditions': ['default', 'opt'],
                  'itpc_basis': 'GT-phase ITPC, test trials, per eval point', 'errors_unit': 'radians'},
       'group_stats': group_stats,
       'optimizer_summary': optimizer_summary,
       'subjects': results}
outpath = os.path.join(OUTDIR, RESULTS_FILENAME)
with open(outpath, 'w') as f:
    json.dump(out, f, indent=2)
print(f"\nSaved -> {outpath}")
print("Each subject holds default + primary-optimizer metrics plus detailed optimizer_runs histories.")


In [ ]:
# ---- CALIBRATION CONVERGENCE PLOT ----
# This plot compares sample efficiency on CAL only. Use the saved held-out TEST metrics to
# compare generalization; do not interpret the calibration minimum as test performance.
import matplotlib.pyplot as plt

if results:
    fig, ax = plt.subplots(figsize=(8, 5))
    trial_axis = np.arange(1, N_BAYES_CALLS + 1)
    for optimizer_name in OPTIMIZERS_TO_RUN:
        curves = np.array([
            [row['best_objective'] for row in r['optimizer_runs'][optimizer_name]['convergence_history']]
            for r in results
        ], dtype=float)
        mean_curve = curves.mean(axis=0)
        sem_curve = curves.std(axis=0, ddof=1) / np.sqrt(curves.shape[0]) if curves.shape[0] > 1 else np.zeros_like(mean_curve)
        ax.plot(trial_axis, mean_curve, linewidth=2, label=optimizer_name)
        ax.fill_between(trial_axis, mean_curve - sem_curve, mean_curve + sem_curve, alpha=0.18)
    ax.axvline(N_INITIAL, color='0.4', linestyle='--', linewidth=1, label='end startup')
    ax.set(xlabel='Objective evaluation', ylabel='Best calibration MACE so far (deg)',
           title='BOB theta optimizer convergence')
    ax.grid(alpha=0.25); ax.legend(); fig.tight_layout()
    figpath = os.path.join(OUTDIR, CONVERGENCE_FIGURE_FILENAME)
    fig.savefig(figpath, dpi=180, bbox_inches='tight')
    plt.show()
    print(f"Saved -> {figpath}")
